## 23_figures
  Draws the figures from saved result tables only; no computation is rerun.
  Output: figures/23_figures/, selected figures in plots/.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ---- Config (all names and paths live here) ----
NOTEBOOK = "23_figures"
RESULTS_19 = "results/core_scaling_decomposition"
FIG_DPI = 200
# -------------------------------------------------

# Project root: the notebook runs from notebooks/ or from the root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FIG_DIR = ROOT / "figures" / NOTEBOOK       # every figure
PLOTS_DIR = ROOT / "plots"                  # article figures only (flat folder)

def save_figure(fig, name, article=False):
    """Save to figures/<notebook>/; also to plots/ when the figure is meant for the article."""
    folders = [FIG_DIR] + ([PLOTS_DIR] if article else [])
    for folder in folders:
        folder.mkdir(parents=True, exist_ok=True)
        path = folder / name
        fig.savefig(path, dpi=FIG_DPI, bbox_inches="tight")
        print("[saved]", path.relative_to(ROOT).as_posix())

print("Figures folder:", FIG_DIR.relative_to(ROOT).as_posix())
print("Article folder:", PLOTS_DIR.relative_to(ROOT).as_posix())

In [ ]:
# ---- Config ----
ROBUST_FILE = "core_scaling_robustness.csv"
SAMPLES = ["pooled_all", "pooled_no_capitals"]
SAMPLE_LABEL = {"pooled_all": "All regions", "pooled_no_capitals": "Without the four capitals"}
FIG1_NAME = "scaling_decomposition_gdp_employment_productivity_2023.png"
Z_95 = 1.96
COLOR_E, COLOR_P = "tab:blue", "tab:orange"
BAR_WIDTH = 0.55
ERRBAR_OFFSET = 0.36            # error bar sits right of the bar, so it does not cross the labels
Y_HEADROOM = 0.55               # space above the tallest bar for labels and the legend
# ----------------

rob = pd.read_csv(ROOT / RESULTS_19 / ROBUST_FILE).set_index("sample").loc[SAMPLES]
assert (rob["identity_diff"].abs() < 1e-9).all(), "beta_Y != beta_E + beta_P"

x = np.arange(len(SAMPLES))
fig, ax = plt.subplots(figsize=(6.5, 6))

ax.bar(x, rob["beta_E"], width=BAR_WIDTH, color=COLOR_E, label=r"$\beta_E$ workplace employment")
ax.bar(x, rob["beta_P"], width=BAR_WIDTH, bottom=rob["beta_E"], color=COLOR_P,
       label=r"$\beta_P$ labour productivity")

# 95% interval of beta_Y (HC3), drawn next to each bar
ax.errorbar(x + ERRBAR_OFFSET, rob["beta_Y"], yerr=Z_95 * rob["se_beta_Y"], fmt="o", ms=4,
            color="black", ecolor="black", capsize=5, lw=1.4,
            label=r"$\beta_Y$ with 95% interval (HC3)")
ax.axhline(1.0, color="0.4", ls="--", lw=1)

for i, (_, r) in enumerate(rob.iterrows()):
    ax.text(i, r["beta_E"] / 2, f"{r['beta_E']:.3f}", ha="center", va="center", color="white")
    ax.text(i, r["beta_E"] + r["beta_P"] / 2, f"{r['beta_P']:.3f}", ha="center", va="center")
    ax.text(i, r["beta_Y"] + 0.03, rf"$\beta_Y$ = {r['beta_Y']:.3f}", ha="center", va="bottom")

ax.set_xticks(x)
ax.set_xticklabels([f"{SAMPLE_LABEL[s]}\n(n = {int(rob.loc[s, 'n'])})" for s in SAMPLES])
ax.set_xlim(-0.5, len(SAMPLES) - 1 + 0.75)
ax.set_ylim(0, rob["beta_Y"].max() + Y_HEADROOM)
ax.set_ylabel("scaling exponent with resident population")
ax.set_title(r"GDP scaling decomposition: $\beta_Y = \beta_E + \beta_P$ (2023)")
ax.legend(loc="upper left", fontsize=9)
fig.tight_layout()

save_figure(fig, FIG1_NAME, article=False)   # supplement only, the article folder gets figure 2
plt.show()

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# ---- Config (file names, model names, labels and colours live here) ----
DELTA_FILE = "delta_comm_2021.csv"
NULL_FILE = "null_model_delta_2021.csv"
CROSSFIT_FILE = "null_model_delta_crossfit_2021.csv"
OBS_SAMPLE, OBS_QUANTITY = "pooled_CZ_PL", "delta_comm"
MODEL_ORDER = ["gravity_exp", "gravity_power", "radiation"]
MODEL_LABEL = {"gravity_exp": "gravity, exponential distance decay",
               "gravity_power": "gravity, power-law distance decay",
               "radiation": "radiation (no free parameter)"}
MODEL_COLOR = {"gravity_exp": "tab:blue", "gravity_power": "tab:blue", "radiation": "tab:orange"}
CROSSFIT_MODELS = ["gravity_exp", "gravity_power"]
CROSSFIT_COLOR = "tab:cyan"
IDENTICAL_TOL = 1e-12                      # the realizations of a null model must be identical
X_MARGIN = 0.03
JITTER = 0.2                               # vertical spread of the cross-fit splits (row units)
SPLIT_SIZE = 70                            # marker area of one split
DECIMALS = {"obs": 3, "null": 3, "cross": 4}
Y_TOP, Y_BOTTOM_PAD = -0.75, 0.45          # axis limits in row units (room above the first row)
FIG3_NAME = "delta_comm_vs_null_models_cz_pl_2021.png"
# ------------------------------------------------------------------------

res = ROOT / RESULTS_19

# Observed delta_comm with its 95% interval
obs_tab = pd.read_csv(res / DELTA_FILE)
obs = obs_tab[(obs_tab["sample"] == OBS_SAMPLE) & (obs_tab["quantity"] == OBS_QUANTITY)]
assert len(obs) == 1, "observed delta_comm row not found"
obs = obs.iloc[0]

# Null models: the realizations must be identical, so one value per model
null_tab = pd.read_csv(res / NULL_FILE)
assert set(null_tab["model"]) == set(MODEL_ORDER), f"models found: {sorted(set(null_tab['model']))}"
spread = null_tab.groupby("model")["delta_null"].agg(lambda s: s.max() - s.min())
assert (spread < IDENTICAL_TOL).all(), "null model realizations differ, review the cell"
null_val = null_tab.groupby("model")["delta_null"].first()

# Cross-fit: five random splits per gravity model
cf_tab = pd.read_csv(res / CROSSFIT_FILE)
assert set(cf_tab["model"]) == set(CROSSFIT_MODELS), f"cross-fit models: {sorted(set(cf_tab['model']))}"
cf_splits = {m: cf_tab.loc[cf_tab["model"] == m, "delta_null"].to_numpy() for m in CROSSFIT_MODELS}

# Rows from top to bottom: observed, three null models, two cross-fit means
rows = [("observed\n(CZ+PL, n = %d)" % int(obs["n"]), "obs", None)]
rows += [(MODEL_LABEL[m], "null", m) for m in MODEL_ORDER]
rows += [("cross-fit: " + MODEL_LABEL[m], "cross", m) for m in CROSSFIT_MODELS]

label_box = dict(boxstyle="round,pad=0.15", facecolor="white", edgecolor="none", alpha=0.85)
fig, ax = plt.subplots(figsize=(9.5, 5.8))
ax.axvspan(obs["ci_low"], obs["ci_high"], color="0.88", zorder=0)
ax.axvline(obs["beta"], color="black", lw=1.2, zorder=1)
ax.axvline(0, color="0.4", ls=":", lw=1.2, zorder=1)

for y, (label, kind, m) in enumerate(rows):
    if kind == "obs":
        ax.errorbar(obs["beta"], y, xerr=[[obs["beta"] - obs["ci_low"]], [obs["ci_high"] - obs["beta"]]],
                    fmt="o", color="black", capsize=5, lw=1.8, ms=7, zorder=3)
        value = obs["beta"]
    elif kind == "null":
        value = null_val[m]
        ax.scatter(value, y, s=90, color=MODEL_COLOR[m], edgecolors="black", zorder=3)
    else:
        pts = cf_splits[m]
        value = pts.mean()
        # Spread the splits vertically so that nearly identical values stay visible
        ys = y + np.linspace(-JITTER, JITTER, len(pts))
        ax.scatter(pts, ys, s=SPLIT_SIZE, facecolors=CROSSFIT_COLOR, edgecolors="black",
                   linewidths=0.8, alpha=0.55, zorder=3)
        ax.scatter(value, y, s=110, marker="D", color=CROSSFIT_COLOR, edgecolors="black", zorder=4)
    top = JITTER * 55 if kind == "cross" else 11            # label height in points
    ax.annotate(f"{value:.{DECIMALS[kind]}f}", (value, y), xytext=(0, top if kind != "cross" else 17),
                textcoords="offset points", ha="center", fontsize=9, bbox=label_box, zorder=5)

ax.set_yticks(range(len(rows)))
ax.set_yticklabels([r[0] for r in rows])
ax.set_ylim(len(rows) - 1 + Y_BOTTOM_PAD, Y_TOP)             # inverted: first row on top
lows = [obs["ci_low"], 0, min(null_val.min(), min(p.min() for p in cf_splits.values()))]
highs = [obs["ci_high"], null_val.max(), max(p.max() for p in cf_splits.values())]
ax.set_xlim(min(lows) - X_MARGIN, max(highs) + X_MARGIN)
ax.set_xlabel(r"$\Delta = \beta_E - \beta_R$   (2021, workplace employment minus resident employment exponent)")
ax.set_title("Commuting-driven employment concentration: observed vs null models")
ax.grid(axis="x", alpha=0.25)

handles = [
    Patch(facecolor="0.88", label="observed 95% interval"),
    Line2D([], [], color="black", lw=1.2, label="observed estimate"),
    Line2D([], [], color="0.4", ls=":", lw=1.2, label="no commuting effect (0)"),
    Line2D([], [], marker="D", ls="", color=CROSSFIT_COLOR, markeredgecolor="black",
           label="cross-fit mean"),
    Line2D([], [], marker="o", ls="", color=CROSSFIT_COLOR, markeredgecolor="black", alpha=0.55,
           label="one cross-fit split (5, spread vertically)"),
]
fig.legend(handles=handles, loc="lower center", ncol=3, fontsize=9, frameon=False)
fig.tight_layout(rect=(0, 0.11, 1, 1))

# Print the numbers used so the figure can be checked against the tables
print(f"observed: {obs['beta']:.4f}  95% [{obs['ci_low']:.4f}, {obs['ci_high']:.4f}]  n = {int(obs['n'])}")
print("null:", {m: round(float(v), 4) for m, v in null_val.items()})
print("cross-fit mean (std):", {m: (round(float(p.mean()), 4), round(float(p.std(ddof=1)), 4))
                                for m, p in cf_splits.items()})

save_figure(fig, FIG3_NAME, article=True)
plt.show()

In [ ]:
import contextlib
import json
import os
import sys

from matplotlib.cm import ScalarMappable
from matplotlib.collections import PolyCollection
from matplotlib.colors import Normalize

# ---- Config (file names, labels and colours live here) ----
NUTS3_GEO = ROOT / "data" / "raw" / "nuts3" / "NUTS_RG_03M_2024_4326.geojson"
SPATIAL_DIR = ROOT / "results" / "spatial_residuals"
RESID_FILE = "residuals_2023.csv"
MORAN_FILE = "moran_2023.csv"
COL_ID, COL_COUNTRY = "nuts3_id", "country_code"
COUNTRIES = ["CZ", "HU", "PL", "SK"]
GEO_COUNTRY_COL, GEO_ID_COL = "CNTR_CODE", "NUTS_ID"
CRS_SRC, CRS_PLOT = "EPSG:4326", "EPSG:3035"
WEIGHTS = "queen"
# (Moran table model, residual column, panel title)
PANELS = [("beta_Y", "resid_beta_Y", "GDP residual"),
          ("beta_E", "resid_beta_E", "Workplace employment residual"),
          ("beta_P", "resid_beta_P", "GDP per employed person residual")]
CMAP = "RdBu_r"
EDGE_COLOR, EDGE_WIDTH = "0.35", 0.3
TOP_N = 3                                   # regions printed per panel on each side
FIG7_NAME = "scaling_residual_maps_full_2023.png"
# -----------------------------------------------------------

@contextlib.contextmanager
def quiet_stderr():
    """Silence C-level stderr (library messages can contain absolute paths)."""
    sys.stderr.flush()
    saved = os.dup(2)
    devnull = os.open(os.devnull, os.O_WRONLY)
    try:
        os.dup2(devnull, 2)
        yield
    finally:
        os.dup2(saved, 2)
        os.close(saved)
        os.close(devnull)

def to_polygons(geometry):
    """Return a list of polygons; a polygon is a list of rings (N x 2 float arrays)."""
    if geometry["type"] == "Polygon":
        polys = [geometry["coordinates"]]
    elif geometry["type"] == "MultiPolygon":
        polys = geometry["coordinates"]
    else:
        raise ValueError(geometry["type"])
    return [[np.asarray(ring, dtype=float)[:, :2] for ring in poly] for poly in polys]

def ring_area(ring):
    """Absolute polygon area of one ring (shoelace formula)."""
    x, y = ring[:, 0], ring[:, 1]
    return 0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))

# Residuals and the Moran table
resid = pd.read_csv(SPATIAL_DIR / RESID_FILE, dtype={COL_ID: str}).set_index(COL_ID)
moran = pd.read_csv(SPATIAL_DIR / MORAN_FILE).set_index("model")
assert set(resid[COL_COUNTRY]) == set(COUNTRIES), f"countries found: {sorted(set(resid[COL_COUNTRY]))}"
assert (moran["weights"] == WEIGHTS).all() and (moran["n"] == len(resid)).all(), "Moran table does not match the residuals"
print(f"regions: {len(resid)} | per country: {resid[COL_COUNTRY].value_counts().sort_index().to_dict()}")

# NUTS3 polygons of the four countries, reprojected to EPSG:3035
with open(NUTS3_GEO, encoding="utf-8") as f:
    feats = [ft for ft in json.load(f)["features"] if ft["properties"][GEO_COUNTRY_COL] in COUNTRIES]
try:
    with quiet_stderr():
        from pyproj import Transformer
        transformer = Transformer.from_crs(CRS_SRC, CRS_PLOT, always_xy=True)
except Exception as e:
    raise RuntimeError(f"pyproj import failed: {type(e).__name__}") from None
poly = {}
for ft in feats:
    parts = to_polygons(ft["geometry"])
    poly[ft["properties"][GEO_ID_COL]] = [
        [np.column_stack(transformer.transform(r[:, 0], r[:, 1])) for r in part] for part in parts]
del feats

# The 2024 edition must contain exactly the regions of the residual table
only_geo, only_res = sorted(set(poly) - set(resid.index)), sorted(set(resid.index) - set(poly))
print(f"NUTS3 polygons: {len(poly)} | only in the polygons: {only_geo} | only in the residuals: {only_res}")
assert not only_geo and not only_res, "region codes differ between the 2024 polygons and the residual table"

# Draw the largest polygons first, so enclaves (e.g. a capital inside its surrounding region) stay visible
items = sorted(((k, part[0]) for k, parts in poly.items() for part in parts),
               key=lambda kr: -ring_area(kr[1]))
keys, rings = [k for k, _ in items], [r for _, r in items]
xy = np.vstack(rings)

cmap = plt.get_cmap(CMAP)
fig, axes = plt.subplots(1, 3, figsize=(18, 6.6))
for ax, (model, col, title) in zip(axes, PANELS):
    values = resid[col]
    vmax = float(values.abs().max())
    norm = Normalize(-vmax, vmax)                      # symmetric around zero: red above, blue below the line
    ax.add_collection(PolyCollection(rings, facecolors=[cmap(norm(values[k])) for k in keys],
                                     edgecolors=EDGE_COLOR, linewidths=EDGE_WIDTH))
    ax.set_xlim(xy[:, 0].min(), xy[:, 0].max())
    ax.set_ylim(xy[:, 1].min(), xy[:, 1].max())
    ax.set_aspect("equal")
    ax.axis("off")
    m = moran.loc[model]
    ax.set_title(f"{title}\nMoran's I = {m['moran_I']:.3f}, pseudo p = {m['p_sim']:.4f}")
    cb = fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), ax=ax, orientation="horizontal",
                      fraction=0.046, pad=0.03)
    cb.set_label("residual (log units, country fixed effects)")

    # Numbers behind the colours, to check against the tables
    order = values.sort_values()
    print(f"{model}: Moran I {m['moran_I']:.4f} (p {m['p_sim']:.4f}) | "
          f"highest {[(k, round(float(v), 3)) for k, v in order.tail(TOP_N)[::-1].items()]} | "
          f"lowest {[(k, round(float(v), 3)) for k, v in order.head(TOP_N).items()]}")

# The residuals come from a model with country fixed effects, so their country means should be ~0
print("largest absolute country mean of the residuals:",
      round(float(resid.groupby(COL_COUNTRY)[[c for _, c, _ in PANELS]].mean().abs().max().max()), 10))

fig.suptitle("Scaling residuals with resident population, 2023 (country fixed effects)")
fig.tight_layout()

save_figure(fig, FIG7_NAME, article=False)        # supplement only
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

# ---- config (uses `paired`, `summary`, `MODELS`, `SAMPLES`, `FIG_DIRS`, `Y_VARIABLE` from the previous cell) ----
GRID = 25                      # bins per axis of the 2D histogram
SIGMA = 1.5                    # Gaussian smoothing, in bins
LEVELS = [1.0, 1.2, 1.5, 2.0, 2.5, 3.0]   # density relative to independence (1 = none)
CMAPS = {"gravity_power": "Blues", "gravity_exp": "Oranges"}
FILE_NAME_DENSITY = "b_beta_comovement_density_gdp_total_pl_2023.png"

edges = np.linspace(0, 1, GRID + 1)
centers = (edges[:-1] + edges[1:]) / 2

fig, axes = plt.subplots(len(MODELS), len(SAMPLES), figsize=(8, 7), sharex=True, sharey=True)
for i, model in enumerate(MODELS):
    for j, sample in enumerate(SAMPLES):
        ax, m = axes[i, j], paired[(sample, model)]
        hist, _, _ = np.histogram2d(m["rank_beta"], m["rank_b"], bins=[edges, edges])
        dens = gaussian_filter(hist, sigma=SIGMA, mode="nearest")
        dens = dens / dens.mean()                     # uniform (independent) density = 1
        cs = ax.contourf(centers, centers, dens.T, levels=LEVELS + [dens.max() + 1e-9],
                         cmap=CMAPS[model], extend="neither")
        ax.contour(centers, centers, dens.T, levels=LEVELS, colors="white", linewidths=0.4)
        ax.plot([0, 1], [0, 1], "--", color="black", lw=0.8)   # perfect co-movement, not b = beta
        rho = summary[(summary["sample"] == sample) & (summary["model"] == model)]["spearman"].iloc[0]
        ax.text(0.04, 0.95, f"Spearman = {rho:.2f}\nn = {len(m)}", transform=ax.transAxes, va="top", fontsize=9,
                bbox=dict(facecolor="white", alpha=0.8, edgecolor="none"))
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        if i == 0:
            ax.set_title(f"PL: {SAMPLES[sample]}", fontsize=10)
        if j == 0:
            ax.set_ylabel(f"rank of b, {MODELS[model]}")
        if i == len(MODELS) - 1:
            ax.set_xlabel(f"rank of beta ({Y_VARIABLE})")
    cbar = fig.colorbar(cs, ax=axes[i, :], fraction=0.04, pad=0.03)
    cbar.set_label("density relative to independence (1 = none)", fontsize=8)

for d in FIG_DIRS:
    d.mkdir(parents=True, exist_ok=True)
    fig.savefig(d / FILE_NAME_DENSITY, dpi=200, bbox_inches="tight")
    print("saved:", (d / FILE_NAME_DENSITY).as_posix())
plt.show()